# Ricci Finance V16 — Automatic Multi-Sector GNN

V16 extends V15 from one manual sector per ticker to normalized multi-sector objects.

**V16 objects**

- official sector and industry;
- automatically inferred business themes;
- optional manual overrides;
- normalized weights whose sum is exactly 1;
- weighted multi-hot GNN node features;
- primary-sector compatibility for the V15 Galaxy and network plots.

## 繁體中文

V16 將 V15 的單一人工產業標籤擴充為自動偵測的多產業物件。每檔股票可同時屬於多個產業與投資主題，而權重總和固定為 1。

## 1. Imports and reproducibility

The example uses only `numpy`, `pandas`, `networkx`, `matplotlib`, and `torch`. It does not require PyTorch Geometric.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Mapping, Sequence

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Automatic V16 sector profiles / 自動產業資料

The notebook uses offline company descriptions by default, so it runs without internet access. Change `USE_YAHOO` to `True` to fetch current metadata through `yfinance`.

Automatic profiles can be combined with manual overrides for special cases.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ricci_finance").exists():
    for candidate in [Path("/mnt/data/v16work"), Path.cwd().parent]:
        if (candidate / "ricci_finance").exists():
            PROJECT_ROOT = candidate
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ricci_finance.sector_objects import (
    build_profile,
    fetch_yfinance_profiles,
    memberships_map,
    primary_sector_map,
)

AUTO_TICKERS = [
    "NVDA", "AMD", "AVGO", "MRVL", "ANET",
    "LITE", "LRCX", "KLAC", "AMAT", "MU",
    "INTC", "AAPL", "META", "IONQ", "QBTS",
]

DEMO_METADATA = {
    "NVDA": ("Technology", "Semiconductors", "GPU artificial intelligence accelerated computing data center networking robotics automotive."),
    "AMD": ("Technology", "Semiconductors", "CPU GPU adaptive computing artificial intelligence accelerator data center."),
    "AVGO": ("Technology", "Semiconductors", "Semiconductor networking broadband wireless infrastructure software data center connectivity."),
    "MRVL": ("Technology", "Semiconductors", "Cloud data infrastructure storage networking optical interconnect automotive semiconductor."),
    "ANET": ("Technology", "Computer Hardware", "Cloud networking Ethernet switching routing artificial intelligence data center network."),
    "LITE": ("Technology", "Communication Equipment", "Optical communications photonics laser transceiver cloud data center network."),
    "LRCX": ("Technology", "Semiconductor Equipment & Materials", "Wafer fabrication equipment deposition etch semiconductor manufacturing."),
    "KLAC": ("Technology", "Semiconductor Equipment & Materials", "Process control inspection metrology semiconductor manufacturing."),
    "AMAT": ("Technology", "Semiconductor Equipment & Materials", "Semiconductor equipment deposition materials engineering display manufacturing."),
    "MU": ("Technology", "Semiconductors", "DRAM NAND flash storage high bandwidth memory data center automotive."),
    "INTC": ("Technology", "Semiconductors", "CPU foundry data center artificial intelligence accelerator networking."),
    "AAPL": ("Technology", "Consumer Electronics", "Smartphone personal computer tablet wearable digital services."),
    "META": ("Communication Services", "Internet Content & Information", "Social media online advertising artificial intelligence virtual reality internet platform."),
    "IONQ": ("Technology", "Computer Hardware", "Trapped ion quantum computer quantum computing cloud access service."),
    "QBTS": ("Technology", "Computer Hardware", "Quantum annealing quantum computing software cloud quantum service."),
}

MANUAL_OVERRIDES = {
    "IONQ": {"QuantumComputing": 0.80, "Cloud": 0.20},
    "QBTS": {"QuantumComputing": 0.85, "AI": 0.15},
}

USE_YAHOO = False

if USE_YAHOO:
    V16_PROFILES = fetch_yfinance_profiles(
        AUTO_TICKERS,
        overrides=MANUAL_OVERRIDES,
    )
else:
    V16_PROFILES = {}
    for ticker in AUTO_TICKERS:
        sector, industry, description = DEMO_METADATA[ticker]
        V16_PROFILES[ticker] = build_profile(
            ticker,
            official_sector=sector,
            official_industry=industry,
            description=description,
            manual_memberships=MANUAL_OVERRIDES.get(ticker),
        )

V16_SECTOR_WEIGHTS = memberships_map(V16_PROFILES)
V16_PRIMARY_SECTORS = primary_sector_map(V16_PROFILES)

profile_rows = []
for ticker, profile in V16_PROFILES.items():
    profile.validate()
    profile_rows.append({
        "ticker": ticker,
        "official_sector": profile.official_sector,
        "official_industry": profile.official_industry,
        "primary_sector": profile.primary_sector,
        "source": profile.source,
        "weight_sum": sum(profile.memberships.values()),
        "memberships": ", ".join(
            f"{label}={weight:.3f}"
            for label, weight in sorted(
                profile.memberships.items(),
                key=lambda item: item[1],
                reverse=True,
            )
        ),
    })

v16_profile_table = pd.DataFrame(profile_rows)
display(v16_profile_table)
assert np.allclose(v16_profile_table["weight_sum"], 1.0)

## 3. Reproducible GNN training example

The original compact dataset remains below so the complete GNN can train quickly.

For production, replace:

```python
RAW_SECTOR_WEIGHTS = {...}
```

with:

```python
RAW_SECTOR_WEIGHTS = V16_SECTOR_WEIGHTS
```

In [ ]:
RAW_SECTOR_WEIGHTS = {
    "NVDA": {"Semiconductors": 6.0, "AI Infrastructure": 3.0, "Networking": 1.0},
    "AMD":  {"Semiconductors": 8.0, "AI Infrastructure": 2.0},
    "MRVL": {"Semiconductors": 5.0, "Networking": 3.0, "AI Infrastructure": 2.0},
    "ANET": {"Networking": 7.0, "AI Infrastructure": 3.0},
    "LITE": {"Networking": 8.0, "Optical": 2.0},
    "MU":   {"Memory": 9.0, "AI Infrastructure": 1.0},
    "AMAT": {"Equipment": 8.5, "Semiconductors": 1.5},
    "LRCX": {"Equipment": 9.0, "Semiconductors": 1.0},
    "IONQ": {"Quantum Computing": 7.0, "Cloud": 2.0, "AI Infrastructure": 1.0},
    "QBTS": {"Quantum Computing": 8.0, "Cloud": 2.0},
    "META": {"Internet": 6.0, "AI Infrastructure": 3.0, "Advertising": 1.0},
    "AAPL": {"Consumer Electronics": 6.0, "Services": 3.0, "Semiconductors": 1.0},
}

## 3. Normalize each ticker independently

Rules:

- Negative values are rejected by default.
- Missing or zero-total memberships fall back to `Other: 1.0`.
- The result for each ticker sums to 1 within numerical tolerance.
- An optional `top_k` can retain only the largest memberships before re-normalizing.

In [ ]:
def normalize_sector_weights(
    raw: Mapping[str, Mapping[str, float] | str],
    *,
    top_k: int | None = None,
    reject_negative: bool = True,
    fallback_sector: str = "Other",
) -> dict[str, dict[str, float]]:
    """Normalize sector memberships separately for every ticker."""
    normalized: dict[str, dict[str, float]] = {}

    for ticker_raw, memberships_raw in raw.items():
        ticker = str(ticker_raw).strip().upper()
        if not ticker:
            raise ValueError("Ticker names must be non-empty")

        # Backward compatibility: {"MU": "Memory"}
        if isinstance(memberships_raw, str):
            memberships = {memberships_raw.strip(): 1.0}
        else:
            memberships = {
                str(sector).strip(): float(weight)
                for sector, weight in memberships_raw.items()
                if str(sector).strip()
            }

        if reject_negative and any(weight < 0 for weight in memberships.values()):
            bad = {s: w for s, w in memberships.items() if w < 0}
            raise ValueError(f"{ticker} has negative sector weights: {bad}")

        # If negatives are allowed, clip them to zero.
        memberships = {s: max(0.0, w) for s, w in memberships.items()}

        if top_k is not None:
            if top_k < 1:
                raise ValueError("top_k must be at least 1")
            memberships = dict(
                sorted(memberships.items(), key=lambda item: item[1], reverse=True)[:top_k]
            )

        total = float(sum(memberships.values()))
        if not np.isfinite(total) or total <= 0:
            normalized[ticker] = {fallback_sector: 1.0}
        else:
            normalized[ticker] = {
                sector: weight / total
                for sector, weight in memberships.items()
                if weight > 0
            }

    return normalized


def validate_normalized_weights(
    memberships: Mapping[str, Mapping[str, float]],
    atol: float = 1e-8,
) -> pd.DataFrame:
    rows = []
    for ticker, weights in memberships.items():
        total = float(sum(weights.values()))
        rows.append({
            "ticker": ticker,
            "sector_count": len(weights),
            "weight_sum": total,
            "minimum_weight": min(weights.values()),
            "valid": bool(np.isclose(total, 1.0, atol=atol) and min(weights.values()) >= 0),
        })
    return pd.DataFrame(rows).set_index("ticker").sort_index()

SECTOR_WEIGHTS = normalize_sector_weights(RAW_SECTOR_WEIGHTS)
validation = validate_normalized_weights(SECTOR_WEIGHTS)
validation

In [ ]:
sector_weight_table = (
    pd.DataFrame(SECTOR_WEIGHTS)
    .T
    .fillna(0.0)
    .sort_index()
)

assert validation["valid"].all()
assert np.allclose(sector_weight_table.sum(axis=1).to_numpy(), 1.0)

sector_weight_table.round(3)

## 4. Visual check of normalized memberships

Every horizontal row represents one ticker. Its stacked sector weights sum to exactly 1.

In [ ]:
ax = sector_weight_table.plot(kind="barh", stacked=True, figsize=(11, 7))
ax.set_title("Normalized multi-sector memberships")
ax.set_xlabel("Membership weight; each ticker sums to 1")
ax.set_ylabel("Ticker")
ax.legend(title="Sector", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 5. Generate synthetic V15-style graph snapshots

This demonstration creates a chronological sequence of ticker networks. Nodes contain V15-like numerical attributes, and edges contain correlation-like weights.

The target has three market regimes:

- `0`: calm
- `1`: sector rotation
- `2`: market stress

The synthetic data is only for testing the multi-sector GNN mechanism. Replace these snapshots and labels with V15's actual `aligned_frames` and HMM regimes in production.

In [ ]:
TICKERS = list(SECTOR_WEIGHTS)
N_SNAPSHOTS = 72
REGIME_NAMES = {0: "Calm", 1: "Sector rotation", 2: "Market stress"}


def shared_sector_similarity(ticker_a: str, ticker_b: str) -> float:
    wa = SECTOR_WEIGHTS[ticker_a]
    wb = SECTOR_WEIGHTS[ticker_b]
    sectors = set(wa) | set(wb)
    return float(sum(min(wa.get(s, 0.0), wb.get(s, 0.0)) for s in sectors))


def make_synthetic_snapshot(t: int, rng: np.random.Generator) -> tuple[nx.Graph, int]:
    # Chronological regime blocks with repeated transitions.
    phase = t % 24
    regime = 0 if phase < 9 else (1 if phase < 17 else 2)

    G = nx.Graph(snapshot=t, regime=regime)

    for ticker in TICKERS:
        ai_weight = SECTOR_WEIGHTS[ticker].get("AI Infrastructure", 0.0)
        quantum_weight = SECTOR_WEIGHTS[ticker].get("Quantum Computing", 0.0)

        base_vol = [0.012, 0.022, 0.045][regime]
        volatility = base_vol * (1 + 0.8 * quantum_weight) + rng.normal(0, 0.0015)
        momentum = [0.010, 0.003, -0.025][regime] + 0.018 * ai_weight + rng.normal(0, 0.008)
        capital_share = max(0.001, rng.lognormal(mean=-2.4 + 0.15 * ai_weight, sigma=0.35))
        node_curvature = [0.20, 0.02, -0.22][regime] + rng.normal(0, 0.05)

        G.add_node(
            ticker,
            degree_proxy=0.0,  # replaced after edges are built
            capital_share=capital_share,
            ricciCurvature=node_curvature,
            momentum=momentum,
            volatility=max(volatility, 0.001),
            volume_change=rng.normal([0.02, 0.10, 0.28][regime], 0.08),
        )

    for i, u in enumerate(TICKERS):
        for v in TICKERS[i + 1:]:
            overlap = shared_sector_similarity(u, v)
            market_component = [0.10, 0.24, 0.52][regime]
            corr = market_component + 0.50 * overlap + rng.normal(0, 0.08)
            corr = float(np.clip(corr, -0.20, 0.95))

            # Sparse in calm periods; denser under stress.
            threshold = [0.42, 0.38, 0.32][regime]
            if corr > threshold:
                G.add_edge(
                    u,
                    v,
                    correlation=corr,
                    distance=float(np.sqrt(max(0.0, 2.0 * (1.0 - corr)))),
                    ricciCurvature=float([0.14, 0.00, -0.18][regime] + rng.normal(0, 0.06)),
                )

    for ticker in G.nodes:
        G.nodes[ticker]["degree_proxy"] = float(G.degree(ticker))

    return G, regime

rng = np.random.default_rng(SEED)
created = [make_synthetic_snapshot(t, rng) for t in range(N_SNAPSHOTS)]
graphs = [item[0] for item in created]
labels = np.array([item[1] for item in created], dtype=np.int64)

print("Snapshots:", len(graphs))
print("Regime counts:", pd.Series(labels).map(REGIME_NAMES).value_counts().to_dict())
print("First graph:", graphs[0])

## 6. Convert a graph to dense GCN tensors

Each node receives six numerical V15-style features plus one feature per sector.

For a ticker belonging to several sectors, its sector part is a **weighted multi-hot vector**. Because each row is normalized, the sector-feature sum is 1 rather than the number of sectors.

In [ ]:
NUMERIC_FEATURES = (
    "degree_proxy",
    "capital_share",
    "ricciCurvature",
    "momentum",
    "volatility",
    "volume_change",
)


def build_sector_vocabulary(
    memberships: Mapping[str, Mapping[str, float]],
) -> dict[str, int]:
    names = sorted({sector for weights in memberships.values() for sector in weights})
    return {sector: i for i, sector in enumerate(names)}


def graph_to_dense_multisector(
    G: nx.Graph,
    nodes: Sequence[str],
    sector_weights: Mapping[str, Mapping[str, float]],
    sector_vocab: Mapping[str, int],
) -> tuple[torch.Tensor, torch.Tensor]:
    node_index = {node: i for i, node in enumerate(nodes)}
    n_nodes = len(nodes)
    n_features = len(NUMERIC_FEATURES) + len(sector_vocab)

    adjacency = np.zeros((n_nodes, n_nodes), dtype=np.float32)
    features = np.zeros((n_nodes, n_features), dtype=np.float32)

    for node, i in node_index.items():
        if node not in G:
            continue

        attrs = G.nodes[node]
        features[i, :len(NUMERIC_FEATURES)] = [
            float(attrs.get(name, 0.0)) for name in NUMERIC_FEATURES
        ]

        memberships = sector_weights.get(str(node).upper(), {"Other": 1.0})
        for sector, weight in memberships.items():
            if sector in sector_vocab:
                features[i, len(NUMERIC_FEATURES) + sector_vocab[sector]] = float(weight)

    for u, v, attrs in G.edges(data=True):
        if u in node_index and v in node_index:
            weight = abs(float(attrs.get("correlation", 1.0)))
            adjacency[node_index[u], node_index[v]] = weight
            adjacency[node_index[v], node_index[u]] = weight

    # Add self-loops and symmetric GCN normalization: D^(-1/2) A D^(-1/2)
    adjacency += np.eye(n_nodes, dtype=np.float32)
    degree = np.maximum(adjacency.sum(axis=1), 1e-8)
    inv_sqrt_degree = 1.0 / np.sqrt(degree)
    adjacency = inv_sqrt_degree[:, None] * adjacency * inv_sqrt_degree[None, :]

    return torch.tensor(features), torch.tensor(adjacency)

NODES = sorted(TICKERS)
SECTOR_VOCAB = build_sector_vocabulary(SECTOR_WEIGHTS)
PAIRS = [
    graph_to_dense_multisector(G, NODES, SECTOR_WEIGHTS, SECTOR_VOCAB)
    for G in graphs
]

print("Node feature tensor:", PAIRS[0][0].shape)
print("Normalized adjacency:", PAIRS[0][1].shape)
print("Numeric features:", len(NUMERIC_FEATURES))
print("Sector features:", len(SECTOR_VOCAB))

In [ ]:
feature_names = list(NUMERIC_FEATURES) + [
    sector for sector, _ in sorted(SECTOR_VOCAB.items(), key=lambda item: item[1])
]
first_features = pd.DataFrame(PAIRS[0][0].numpy(), index=NODES, columns=feature_names)

sector_columns = feature_names[len(NUMERIC_FEATURES):]
sector_sums = first_features[sector_columns].sum(axis=1)
assert np.allclose(sector_sums.to_numpy(), 1.0)

first_features.loc[["NVDA", "MRVL", "IONQ"], sector_columns].round(3)

## 7. Dense graph-level GCN

This follows V15's graph-classification structure:

1. two graph-convolution layers,
2. mean pooling across nodes,
3. one output vector per graph snapshot,
4. chronological 70/30 train/test split,
5. class-weighted cross-entropy.

In [ ]:
@dataclass
class GNNResult:
    labels: np.ndarray
    predictions: np.ndarray
    probabilities: np.ndarray
    train_indices: np.ndarray
    test_indices: np.ndarray
    accuracy: float
    balanced_accuracy: float
    losses: list[float]
    device: str
    class_weights: np.ndarray


class DenseGCN(nn.Module):
    def __init__(self, input_features: int, hidden: int, classes: int):
        super().__init__()
        self.layer1 = nn.Linear(input_features, hidden)
        self.layer2 = nn.Linear(hidden, hidden)
        self.output = nn.Linear(hidden, classes)

    def forward(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        h = torch.relu(self.layer1(a @ x))
        h = torch.relu(self.layer2(a @ h))
        graph_embedding = h.mean(dim=0)
        return self.output(graph_embedding)


def train_multisector_gcn(
    pairs: Sequence[tuple[torch.Tensor, torch.Tensor]],
    labels: Sequence[int],
    *,
    epochs: int = 160,
    hidden: int = 32,
    learning_rate: float = 0.004,
    random_state: int = 42,
) -> GNNResult:
    torch.manual_seed(random_state)
    np.random.seed(random_state)

    y = np.asarray(labels, dtype=np.int64).reshape(-1)
    if len(pairs) != len(y):
        raise ValueError("Graph/label length mismatch")
    if len(pairs) < 10:
        raise ValueError("At least 10 graph snapshots are required")

    classes = np.unique(y)
    remap = {original: new for new, original in enumerate(classes)}
    y_remapped = np.array([remap[value] for value in y], dtype=np.int64)

    split = max(2, min(len(y) - 2, int(len(y) * 0.70)))
    train_indices = np.arange(split)
    test_indices = np.arange(split, len(y))

    counts = np.bincount(y_remapped[train_indices], minlength=len(classes))
    class_weights = len(train_indices) / (len(classes) * np.maximum(counts, 1))

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = DenseGCN(pairs[0][0].shape[1], hidden, len(classes)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    loss_function = nn.CrossEntropyLoss(
        weight=torch.tensor(class_weights, dtype=torch.float32, device=device)
    )

    losses: list[float] = []
    model.train()
    for _ in range(int(epochs)):
        optimizer.zero_grad()
        logits = torch.stack([
            model(x.to(device), a.to(device)) for x, a in pairs[:split]
        ])
        target = torch.tensor(y_remapped[:split], dtype=torch.long, device=device)
        loss = loss_function(logits, target)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))

    model.eval()
    with torch.no_grad():
        logits = torch.stack([
            model(x.to(device), a.to(device)) for x, a in pairs
        ])
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()
        predicted_remapped = probabilities.argmax(axis=1)

    predictions = np.array([classes[i] for i in predicted_remapped])
    accuracy = float((predictions[test_indices] == y[test_indices]).mean())

    recalls = []
    for cls in np.unique(y[test_indices]):
        mask = y[test_indices] == cls
        recalls.append(float((predictions[test_indices][mask] == cls).mean()))
    balanced_accuracy = float(np.mean(recalls)) if recalls else float("nan")

    return GNNResult(
        labels=y,
        predictions=predictions,
        probabilities=probabilities,
        train_indices=train_indices,
        test_indices=test_indices,
        accuracy=accuracy,
        balanced_accuracy=balanced_accuracy,
        losses=losses,
        device=device,
        class_weights=class_weights,
    )

result = train_multisector_gcn(PAIRS, labels)
print("Device:", result.device)
print("Test accuracy:", round(result.accuracy, 3))
print("Balanced accuracy:", round(result.balanced_accuracy, 3))
print("Class weights:", np.round(result.class_weights, 3))

## 8. GNN input-network plot

This interactive Plotly view shows the **same ticker graph and attributes supplied to the GCN**. Choose any snapshot index. Node color represents the primary sector, node size represents capital share, edge width represents absolute correlation, and edge style separates positive and negative Ricci curvature. Hovering over a node displays all normalized multi-sector weights, whose sum is one for each ticker.


In [ ]:
import plotly.graph_objects as go


def primary_sector(ticker: str) -> str:
    weights = SECTOR_WEIGHTS.get(ticker, {"Other": 1.0})
    return max(weights, key=weights.get)


def format_memberships(ticker: str) -> str:
    weights = SECTOR_WEIGHTS.get(ticker, {"Other": 1.0})
    ordered = sorted(weights.items(), key=lambda item: item[1], reverse=True)
    return "<br>".join(f"{sector}: {weight:.3f}" for sector, weight in ordered)


def gnn_network_figure(
    snapshot_index: int,
    *,
    layout_seed: int = 42,
    dimensions: int = 2,
) -> go.Figure:
    """Plot one graph snapshot used by the GCN.

    Parameters
    ----------
    snapshot_index:
        Position in ``graphs`` and ``result``.
    layout_seed:
        Fixed seed keeps positions stable between reruns.
    dimensions:
        Currently 2. Reserved for a future 3-D implementation.
    """
    if dimensions != 2:
        raise ValueError("This example currently supports dimensions=2")
    if not 0 <= snapshot_index < len(graphs):
        raise IndexError(f"snapshot_index must be from 0 to {len(graphs) - 1}")

    G = graphs[snapshot_index]
    pos = nx.spring_layout(G, seed=layout_seed, weight="correlation", iterations=120)

    # Use separate traces so positive and negative Ricci edges are distinguishable.
    edge_groups = {
        "Positive Ricci edge": {"x": [], "y": [], "width": []},
        "Negative Ricci edge": {"x": [], "y": [], "width": []},
    }

    edge_hover_x, edge_hover_y, edge_hover_text = [], [], []
    for u, v, attrs in G.edges(data=True):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        corr = float(attrs.get("correlation", 0.0))
        ricci = float(attrs.get("ricciCurvature", 0.0))
        group = "Positive Ricci edge" if ricci >= 0 else "Negative Ricci edge"
        edge_groups[group]["x"] += [x0, x1, None]
        edge_groups[group]["y"] += [y0, y1, None]
        edge_groups[group]["width"].append(0.6 + 3.2 * abs(corr))

        # Invisible midpoint markers provide precise edge hover information.
        edge_hover_x.append((x0 + x1) / 2)
        edge_hover_y.append((y0 + y1) / 2)
        edge_hover_text.append(
            f"{u} — {v}<br>Correlation: {corr:.3f}"
            f"<br>Distance: {float(attrs.get('distance', np.nan)):.3f}"
            f"<br>Ricci curvature: {ricci:.3f}"
        )

    fig = go.Figure()
    edge_dash = {"Positive Ricci edge": "solid", "Negative Ricci edge": "dot"}
    for name, values in edge_groups.items():
        if not values["x"]:
            continue
        # Plotly applies one width per line trace. Use the group mean while exact
        # correlation remains visible in the edge hover markers.
        mean_width = float(np.mean(values["width"])) if values["width"] else 1.0
        fig.add_trace(go.Scatter(
            x=values["x"], y=values["y"],
            mode="lines",
            line={"width": mean_width, "dash": edge_dash[name]},
            hoverinfo="skip",
            name=name,
        ))

    fig.add_trace(go.Scatter(
        x=edge_hover_x,
        y=edge_hover_y,
        mode="markers",
        marker={"size": 10, "opacity": 0.0},
        text=edge_hover_text,
        hovertemplate="%{text}<extra></extra>",
        name="Edge details",
        showlegend=False,
    ))

    sectors = sorted({primary_sector(node) for node in G.nodes})
    sector_to_code = {sector: i for i, sector in enumerate(sectors)}

    node_x, node_y, node_text, node_sizes, node_codes, labels_text = [], [], [], [], [], []
    capital_values = np.array([
        float(G.nodes[node].get("capital_share", 0.0)) for node in G.nodes
    ])
    capital_scale = capital_values / max(float(capital_values.max()), 1e-12)

    for node, scaled_capital in zip(G.nodes, capital_scale):
        x, y = pos[node]
        attrs = G.nodes[node]
        sector = primary_sector(node)
        memberships_sum = sum(SECTOR_WEIGHTS.get(node, {"Other": 1.0}).values())

        node_x.append(x)
        node_y.append(y)
        labels_text.append(node)
        node_codes.append(sector_to_code[sector])
        node_sizes.append(18 + 35 * np.sqrt(max(scaled_capital, 0.0)))
        node_text.append(
            f"<b>{node}</b>"
            f"<br>Primary sector: {sector}"
            f"<br><b>Normalized sector features</b><br>{format_memberships(node)}"
            f"<br>Weight sum: {memberships_sum:.6f}"
            f"<br>Degree: {G.degree(node)}"
            f"<br>Capital share: {float(attrs.get('capital_share', 0.0)):.4f}"
            f"<br>Momentum: {float(attrs.get('momentum', 0.0)):.4f}"
            f"<br>Volatility: {float(attrs.get('volatility', 0.0)):.4f}"
            f"<br>Node Ricci: {float(attrs.get('ricciCurvature', 0.0)):.4f}"
        )

    fig.add_trace(go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=labels_text,
        textposition="top center",
        textfont={"size": 13},
        customdata=node_text,
        hovertemplate="%{customdata}<extra></extra>",
        marker={
            "size": node_sizes,
            "color": node_codes,
            "colorscale": "Turbo",
            "showscale": False,
            "line": {"width": 1.5},
        },
        name="Ticker nodes",
    ))

    true_name = REGIME_NAMES[int(result.labels[snapshot_index])]
    predicted_name = REGIME_NAMES[int(result.predictions[snapshot_index])]
    probabilities = result.probabilities[snapshot_index]
    probability_text = ", ".join(
        f"{REGIME_NAMES[i]}={probability:.3f}"
        for i, probability in enumerate(probabilities)
    )

    fig.update_layout(
        title=(
            f"GNN input network — snapshot {snapshot_index}"
            f"<br><sup>True: {true_name} | Predicted: {predicted_name} | "
            f"{probability_text}</sup>"
        ),
        height=760,
        hovermode="closest",
        margin={"l": 20, "r": 20, "t": 90, "b": 20},
        xaxis={"visible": False},
        yaxis={"visible": False, "scaleanchor": "x", "scaleratio": 1},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02},
        template="plotly_white",
    )
    return fig


# The next cells explicitly render both an interactive Plotly graph and
# a static Matplotlib fallback.


In [ ]:
from IPython.display import display
import plotly.io as pio

# Explicit notebook renderer. `display(fig)` is more reliable than a bare `.show()`
# across JupyterLab, classic Notebook, VS Code, and browser-based notebooks.
pio.renderers.default = "notebook_connected"

SELECTED_SNAPSHOT = int(result.test_indices[-1])
interactive_fig = gnn_network_figure(SELECTED_SNAPSHOT)
display(interactive_fig)


In [ ]:
def plot_gnn_network_static(
    snapshot_index: int,
    *,
    layout_seed: int = 42,
    figsize: tuple[float, float] = (14, 10),
):
    """Static fallback that always renders inline with Matplotlib."""
    if not 0 <= snapshot_index < len(graphs):
        raise IndexError(f"snapshot_index must be from 0 to {len(graphs) - 1}")

    G = graphs[snapshot_index]
    pos = nx.spring_layout(
        G,
        seed=layout_seed,
        weight="correlation",
        iterations=150,
    )

    primary = [primary_sector(str(node)) for node in G.nodes]
    sector_names = sorted(set(primary))
    sector_code = {sector: i for i, sector in enumerate(sector_names)}
    node_colors = [sector_code[sector] for sector in primary]

    capital = np.array([
        float(G.nodes[node].get("capital_share", 0.0))
        for node in G.nodes
    ])
    capital = capital / max(float(capital.max()), 1e-12)
    node_sizes = 900 + 2600 * np.sqrt(np.clip(capital, 0.0, None))

    positive_edges = []
    negative_edges = []
    edge_widths_positive = []
    edge_widths_negative = []

    for u, v, attrs in G.edges(data=True):
        corr = abs(float(attrs.get("correlation", 0.0)))
        ricci = float(attrs.get("ricciCurvature", 0.0))
        width = 0.8 + 4.0 * corr
        if ricci >= 0:
            positive_edges.append((u, v))
            edge_widths_positive.append(width)
        else:
            negative_edges.append((u, v))
            edge_widths_negative.append(width)

    fig, ax = plt.subplots(figsize=figsize)

    if positive_edges:
        nx.draw_networkx_edges(
            G, pos, ax=ax,
            edgelist=positive_edges,
            width=edge_widths_positive,
            alpha=0.55,
            style="solid",
        )

    if negative_edges:
        nx.draw_networkx_edges(
            G, pos, ax=ax,
            edgelist=negative_edges,
            width=edge_widths_negative,
            alpha=0.55,
            style="dashed",
        )

    nodes_artist = nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_size=node_sizes,
        node_color=node_colors,
        cmap=plt.cm.tab20,
        edgecolors="black",
        linewidths=1.2,
    )

    nx.draw_networkx_labels(
        G, pos, ax=ax,
        font_size=10,
        font_weight="bold",
    )

    true_name = REGIME_NAMES[int(result.labels[snapshot_index])]
    predicted_name = REGIME_NAMES[int(result.predictions[snapshot_index])]
    probability_text = ", ".join(
        f"{REGIME_NAMES[i]}={p:.3f}"
        for i, p in enumerate(result.probabilities[snapshot_index])
    )

    ax.set_title(
        f"GNN network — snapshot {snapshot_index}\n"
        f"True: {true_name} | Predicted: {predicted_name} | {probability_text}",
        fontsize=14,
    )
    ax.axis("off")

    # Sector legend
    handles = []
    for sector, code in sector_code.items():
        handles.append(
            plt.Line2D(
                [0], [0],
                marker="o",
                linestyle="",
                markerfacecolor=plt.cm.tab20(code / max(len(sector_names) - 1, 1)),
                markeredgecolor="black",
                markersize=10,
                label=sector,
            )
        )
    handles.extend([
        plt.Line2D([0], [0], linewidth=2, linestyle="solid", label="Ricci ≥ 0"),
        plt.Line2D([0], [0], linewidth=2, linestyle="dashed", label="Ricci < 0"),
    ])
    ax.legend(
        handles=handles,
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        frameon=True,
    )

    plt.tight_layout()
    plt.show()
    return fig


# This cell produces a visible plot even when Plotly rendering is unavailable.
static_fig = plot_gnn_network_static(SELECTED_SNAPSHOT)


## 9. Training and prediction diagnostics

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(result.losses)
plt.title("GCN training loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.tight_layout()
plt.show()

In [ ]:
prediction_table = pd.DataFrame({
    "snapshot": np.arange(len(labels)),
    "actual": pd.Series(result.labels).map(REGIME_NAMES),
    "predicted": pd.Series(result.predictions).map(REGIME_NAMES),
    "split": np.where(np.arange(len(labels)) < result.test_indices[0], "train", "test"),
    "confidence": result.probabilities.max(axis=1),
})

prediction_table.tail(22)

In [ ]:
confusion = pd.crosstab(
    prediction_table.loc[result.test_indices, "actual"],
    prediction_table.loc[result.test_indices, "predicted"],
    rownames=["Actual"],
    colnames=["Predicted"],
    dropna=False,
)
confusion

## 10. Optional comparison: one-hot primary sector versus weighted multi-sector

The function below converts every ticker to only its largest sector. This enables an apples-to-apples comparison while keeping the graph snapshots and GCN architecture unchanged.

In [ ]:
def primary_sector_only(
    memberships: Mapping[str, Mapping[str, float]],
) -> dict[str, dict[str, float]]:
    return {
        ticker: {max(weights, key=weights.get): 1.0}
        for ticker, weights in memberships.items()
    }

PRIMARY_ONLY = primary_sector_only(SECTOR_WEIGHTS)
PRIMARY_VOCAB = build_sector_vocabulary(PRIMARY_ONLY)
PRIMARY_PAIRS = [
    graph_to_dense_multisector(G, NODES, PRIMARY_ONLY, PRIMARY_VOCAB)
    for G in graphs
]

primary_result = train_multisector_gcn(PRIMARY_PAIRS, labels)

comparison = pd.DataFrame({
    "model": ["Weighted multi-sector", "Primary-sector one-hot"],
    "sector_feature_count": [len(SECTOR_VOCAB), len(PRIMARY_VOCAB)],
    "test_accuracy": [result.accuracy, primary_result.accuracy],
    "balanced_accuracy": [result.balanced_accuracy, primary_result.balanced_accuracy],
})
comparison.round(3)

## 11. Drop-in adaptation for V15 `ricci_finance/gnn.py`

The essential V15 changes are:

```python
# Old type
sectors: Dict[str, str]

# New type
sector_weights: Dict[str, Dict[str, float]]
```

Replace the old vocabulary construction:

```python
sec_names = sorted(set(sectors.get(str(n), "Other") for n in nodes))
```

with:

```python
sec_names = sorted({
    sector
    for node in nodes
    for sector in sector_weights.get(str(node), {"Other": 1.0})
})
```

Replace the old one-hot assignment:

```python
sec = sectors.get(str(n), "Other")
X[i, 6 + sector_vocab[sec]] = 1.0
```

with weighted multi-hot assignment:

```python
memberships = sector_weights.get(str(n), {"Other": 1.0})
for sector, weight in memberships.items():
    X[i, 6 + sector_vocab[sector]] = weight
```

Always call `normalize_sector_weights(...)` before constructing GNN tensors.

## 12. Recommended production rules

1. Normalize independently per ticker; do not normalize globally across all tickers.
2. Reject negative or non-finite weights.
3. Use `Other: 1.0` when no valid membership is supplied.
4. Keep sector labels and thematic labels separate when possible; for example, `Semiconductors` is an industry while `AI Infrastructure` is a theme.
5. Save the normalized mapping used in each experiment for reproducibility.
6. Fit normalization and model configuration without using future test labels.
7. Keep Ricci curvature computed on the ticker graph; multi-sector membership is an additional node feature and does not require duplicating ticker nodes.

## 13. Weighted sector momentum / 加權產業動能

\[
M_s = \frac{\sum_i w_{i,s}m_i}{\sum_i w_{i,s}}
\]

In [ ]:
from collections import defaultdict

def weighted_sector_momentum(ticker_momentum, weights):
    numerator = defaultdict(float)
    denominator = defaultdict(float)
    for ticker, value in ticker_momentum.items():
        for sector, weight in weights[ticker].items():
            numerator[sector] += float(weight) * float(value)
            denominator[sector] += float(weight)
    return pd.Series({
        sector: numerator[sector] / denominator[sector]
        for sector in numerator
        if denominator[sector] > 0
    }).sort_values(ascending=False)

demo_momentum = pd.Series({
    ticker: np.random.default_rng(i).normal(0.001, 0.01)
    for i, ticker in enumerate(AUTO_TICKERS)
})

automatic_sector_momentum = weighted_sector_momentum(
    demo_momentum,
    V16_SECTOR_WEIGHTS,
)

display(automatic_sector_momentum.to_frame("weighted_momentum").round(5))
automatic_sector_momentum.plot(kind="bar", figsize=(12, 5), title="V16 weighted sector momentum")
plt.tight_layout()
plt.show()

## 14. Weighted sector capital flow / 加權產業資金流

For edge \((i,j)\):

\[
F_{ab}^{(i,j)} = F_{ij}w_{i,a}w_{j,b}
\]

The total flow is conserved because each ticker's weights sum to one.

In [ ]:
def weighted_sector_flow_matrix(G, weights):
    sectors = sorted({
        sector
        for values in weights.values()
        for sector in values
    })
    matrix = pd.DataFrame(0.0, index=sectors, columns=sectors)
    ticker_total = 0.0

    for u, v, attrs in G.edges(data=True):
        if str(u) not in weights or str(v) not in weights:
            continue
        flow = abs(float(attrs.get("correlation", 0.0)))
        ticker_total += flow
        for source_sector, source_weight in weights[str(u)].items():
            for target_sector, target_weight in weights[str(v)].items():
                matrix.loc[source_sector, target_sector] += (
                    flow * source_weight * target_weight
                )

    return matrix, ticker_total

flow_matrix, ticker_total = weighted_sector_flow_matrix(
    graphs[-1],
    SECTOR_WEIGHTS,
)

print("Ticker-edge flow:", round(ticker_total, 8))
print("Sector flow:", round(float(flow_matrix.values.sum()), 8))
assert np.isclose(flow_matrix.values.sum(), ticker_total)

fig, ax = plt.subplots(figsize=(11, 9))
image = ax.imshow(flow_matrix.values, aspect="auto")
ax.set_xticks(np.arange(len(flow_matrix.columns)))
ax.set_xticklabels(flow_matrix.columns, rotation=50, ha="right")
ax.set_yticks(np.arange(len(flow_matrix.index)))
ax.set_yticklabels(flow_matrix.index)
ax.set_title("V16 weighted sector capital-flow matrix")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

## 15. V16 production integration

Main objects:

```python
V16_PROFILES
V16_SECTOR_WEIGHTS
V16_PRIMARY_SECTORS
```

GNN:

```python
train_gcn_regime(graphs, labels, V16_SECTOR_WEIGHTS)
```

Keep `V16_PRIMARY_SECTORS` for V15-compatible node colors and Galaxy placement.

Recommended explicit cache:

```text
cache/sector_profiles.json
```

V16 preserves the Ricci graph topology while adding automatic, normalized and GNN-ready company context.

## V16.2 — Dynamic sectors, GAT and temporal topology

This update adds six connected analyses:

1. dynamic sector memberships for every rolling graph;
2. 2D/3D GNN graph embeddings using PCA, UMAP or t-SNE;
3. optional GAT attention matrices;
4. temporal Ricci-community animation;
5. comparison of Yahoo sector, detected theme, Ricci community and GNN latent cluster;
6. an animated 3D Galaxy where angle is sector/theme, radius is capital share, and height is Ricci curvature or a GNN latent coordinate.

The Streamlit app exposes these controls directly. The reusable implementation is in `dynamic.py`, `advanced_visualization.py`, and the updated `gnn.py`.

In [ ]:
from ricci_finance.dynamic import (
    build_dynamic_sector_history,
    ricci_communities,
    sector_evolution_table,
)
from ricci_finance.advanced_visualization import (
    community_animation,
    galaxy_animation,
    reduce_embeddings,
    embedding_figure,
)
from ricci_finance.gnn import train_graph_regime

# After aligned_frames and V16_SECTOR_WEIGHTS exist:
# dynamic_history = build_dynamic_sector_history(aligned_frames, V16_SECTOR_WEIGHTS)
# communities = [ricci_communities(frame["graph"]) for frame in aligned_frames]
# result = train_graph_regime(
#     [frame["graph"] for frame in aligned_frames],
#     labels,
#     dynamic_history,
#     model_type="GAT",       # or "GCN"
# )
# reduced = reduce_embeddings(result.graph_embeddings, dimensions=3, method="UMAP")
# embedding_figure(reduced, result.predictions, graph_dates, dimensions=3).show()
# community_animation(aligned_frames, communities).show()
# galaxy_animation(aligned_frames, dynamic_history, result.graph_embeddings).show()